In [16]:
import joblib
import pandas as pd
import numpy as np

scaler = joblib.load("../models/scaler.pkl")
final_feature_names = joblib.load("../models/final_feature_names.pkl")
try:
    preprocessor = joblib.load("../models/house_preprocessor.pkl")
except Exception:
    preprocessor = None

model = joblib.load("../models/ridge_model.pkl")

try:
    X_test_rfe = pd.read_csv("../data/X_test_rfe.csv")
except Exception:
    test_raw = pd.read_csv("../house-prices-advanced-regression-techniques/test.csv").drop(columns=["Id"])
    if preprocessor is None:
        raise RuntimeError("No preprocessor found and no X_test_rfe.csv available.")
    X_test_rfe = preprocessor.transform(test_raw)

X_test_rfe = X_test_rfe.reindex(columns=final_feature_names, fill_value=0)

X_test_scaled_arr = scaler.transform(X_test_rfe)
X_test_scaled = pd.DataFrame(X_test_scaled_arr, columns=final_feature_names, index=X_test_rfe.index)

y_test_pred_log = model.predict(X_test_scaled)
y_test_pred = np.expm1(y_test_pred_log)

test_ids = joblib.load("../models/test_ids.pkl")
submission = pd.DataFrame({"Id": test_ids, "SalePrice": y_test_pred})
submission.to_csv("../data/submission.csv", index=False)
print("Saved submission to ../data/submission.csv")


Saved submission to ../data/submission.csv


In [17]:
import joblib, pandas as pd, numpy as np
X_train_ready = joblib.load("../models/X_train_ready.pkl")
final_cols = joblib.load("../models/final_feature_names.pkl")
X_test_rfe = pd.read_csv("../data/X_test_rfe.csv").reindex(columns=final_cols, fill_value=0)

print("Train cols count:", len(X_train_ready.columns))
print("Test cols count:", len(X_test_rfe.columns))
print("Train-only cols:", sorted(set(X_train_ready.columns) - set(X_test_rfe.columns))[:10])
print("Test-only cols:", sorted(set(X_test_rfe.columns) - set(X_train_ready.columns))[:10])


Train cols count: 100
Test cols count: 100
Train-only cols: []
Test-only cols: []
